# Set Card Classifier — Inference

Standalone inference notebook. No training required — just a trained checkpoint in `checkpoints/`.

**Three modes** (set `INPUT` in the config cell below):
- **Single file** — point at one image
- **Folder** — run on all `.jpg` files in a directory
- **`None`** — randomly sample from `data/raw/` and test augmented variants

**Run order:** `Setup` → `Config` → `Load Model` → any section below

## Setup

In [ ]:
import os
import glob
import random
import sys
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import cv2

# notebooks/ is one level below the project root
sys.path.append(os.path.abspath('..'))

from src.models.multi_head_resnet import MultiHeadResNet, FEATURE_NAMES
from src.models.predictor import predict, predict_batch, INVERSE_LABEL_MAPS
from src.data.set_card_data_pipeline import _LABEL_ALIASES
from src.utils.augmentations import get_train_transforms

if torch.cuda.is_available():
    DEVICE = 'cuda'
elif torch.backends.mps.is_available():
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'

print(f'Device: {DEVICE}')

## Config

Set `INPUT` to control what gets classified:
```python
INPUT = '../data/raw/red_diamond_1_solid.jpg'  # single image
INPUT = '../data/raw/'                          # whole folder
INPUT = None                                    # random sample + augmentation test
```

In [ ]:
# ── Change these ─────────────────────────────────────────────────────────────
INPUT       = None           # file path, folder path, or None
RAW_DIR     = '../data/raw'
N_SAMPLES   = 8              # max images to show (folder / None mode)
N_AUGMENTED = 4              # augmented variants per image (None mode only)
# ─────────────────────────────────────────────────────────────────────────────

## Load Model

Auto-finds the checkpoint with the highest `val_pma` score in `checkpoints/`.

In [ ]:
_ckpts = glob.glob('../checkpoints/best-*.ckpt')
if not _ckpts:
    raise FileNotFoundError('No checkpoint found in checkpoints/. Train the model first.')

_ckpt_path = sorted(
    _ckpts,
    key=lambda p: float(p.split('val_pma=')[1].replace('.ckpt', ''))
)[-1]

model = MultiHeadResNet.load_from_checkpoint(_ckpt_path).to(DEVICE)
model.eval()
print(f'Loaded: {_ckpt_path}')

## Inference

Resolves images based on `INPUT`, runs `predict_batch`, and displays results.
Titles are **green** if all 4 features are correct, **red** if any are wrong (only when ground truth is available from the filename).

In [ ]:
def _parse_ground_truth(path):
    """Returns ground truth dict from filename, or None if not in naming convention."""
    parts = os.path.basename(path).split('_')
    if len(parts) < 4:
        return None
    try:
        raw = {
            'color':   parts[0],
            'shape':   parts[1],
            'number':  parts[2],
            'shading': parts[3].split('.')[0],
        }
        return {f: _LABEL_ALIASES.get(f, {}).get(raw[f], raw[f]) for f in raw}
    except Exception:
        return None

# Resolve image paths
if INPUT is None:
    all_raw = glob.glob(os.path.join(RAW_DIR, '*.jpg'))
    image_paths = random.sample(all_raw, min(N_SAMPLES, len(all_raw)))
    print(f'Random mode: sampled {len(image_paths)} images from {RAW_DIR}')
elif os.path.isfile(INPUT):
    image_paths = [INPUT]
    print(f'Single file: {INPUT}')
elif os.path.isdir(INPUT):
    all_in_dir = glob.glob(os.path.join(INPUT, '*.jpg'))
    image_paths = random.sample(all_in_dir, min(N_SAMPLES, len(all_in_dir)))
    print(f'Folder mode: {len(image_paths)} images from {INPUT}')
else:
    raise ValueError(f'INPUT={INPUT!r} is not a valid file or directory.')

# Run inference
results = predict_batch(model, image_paths)

# Display grid
ncols = min(4, len(image_paths))
nrows = (len(image_paths) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3.5, nrows * 4.5))
axes = np.array(axes).flatten() if len(image_paths) > 1 else [axes]

for ax, path, result in zip(axes, image_paths, results):
    img_rgb = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
    ax.imshow(img_rgb)
    gt = _parse_ground_truth(path)
    lines, all_correct = [], True
    for feature in FEATURE_NAMES:
        pred = result[feature]['prediction']
        conf = result[feature]['confidence']
        if gt:
            ok = pred == gt[feature]
            all_correct = all_correct and ok
            lines.append(f"{feature}: {pred} ({conf:.0%}) {'OK' if ok else f'!! gt:{gt[feature]}'}")
        else:
            lines.append(f'{feature}: {pred} ({conf:.0%})')
    title_color = ('green' if all_correct else 'red') if gt else 'black'
    ax.set_title('\n'.join(lines), fontsize=7, color=title_color)
    ax.axis('off')

for ax in axes[len(image_paths):]:
    ax.axis('off')

plt.suptitle(f'Inference Results — {len(image_paths)} image(s)', fontsize=13)
plt.tight_layout()
plt.show()

## Augmentation Robustness Test

Only runs when `INPUT = None`. Shows the original image alongside `N_AUGMENTED` randomly augmented variants and whether the model still classifies each correctly.

This tests how well the model handles the same card under rotation, brightness shifts, and background changes.

In [ ]:
if INPUT is not None:
    print('Augmentation robustness test only runs when INPUT = None.')
else:
    _MEAN = np.array([0.485, 0.456, 0.406])
    _STD  = np.array([0.229, 0.224, 0.225])
    transform = get_train_transforms()

    def denorm(tensor):
        img = tensor.permute(1, 2, 0).numpy()
        return np.clip(img * _STD + _MEAN, 0, 1)

    n_rows = len(image_paths)
    n_cols = 1 + N_AUGMENTED  # original + augmented variants
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 2.8, n_rows * 3.5))
    if n_rows == 1:
        axes = [axes]

    for row, path in enumerate(image_paths):
        img_rgb = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
        gt = _parse_ground_truth(path)

        # Column 0: original image
        orig_result = predict(model, path)
        all_ok_orig = all(orig_result[f]['prediction'] == gt[f] for f in FEATURE_NAMES) if gt else None
        ax = axes[row][0]
        ax.imshow(img_rgb)
        ax.set_title(
            f"original\n{'OK' if all_ok_orig else '!!'}",
            fontsize=8,
            color=('green' if all_ok_orig else 'red') if all_ok_orig is not None else 'black'
        )
        ax.axis('off')

        # Columns 1..N_AUGMENTED: apply train augmentations, run model directly on tensors
        aug_tensors = [transform(image=img_rgb)['image'] for _ in range(N_AUGMENTED)]
        batch = torch.stack(aug_tensors).to(DEVICE)
        with torch.no_grad():
            logits = model(batch)

        for col, tensor in enumerate(aug_tensors):
            aug_result = {
                f: {
                    'prediction': INVERSE_LABEL_MAPS[f][F.softmax(logits[f][col], dim=0).argmax().item()],
                    'confidence': round(F.softmax(logits[f][col], dim=0).max().item(), 4),
                }
                for f in FEATURE_NAMES
            }
            all_ok = all(aug_result[f]['prediction'] == gt[f] for f in FEATURE_NAMES) if gt else None
            ax = axes[row][col + 1]
            ax.imshow(denorm(tensor.cpu()))
            ax.set_title(
                f"aug {col + 1}\n{'OK' if all_ok else '!!'}",
                fontsize=8,
                color=('green' if all_ok else 'red') if all_ok is not None else 'black'
            )
            ax.axis('off')

    plt.suptitle(f'Augmentation Robustness — {N_AUGMENTED} variants per image', fontsize=13)
    plt.tight_layout()
    plt.show()

## Performance Benchmark

Measures single-image latency and batch throughput. Includes warmup runs to avoid cold-start JIT overhead.

In [ ]:
WARMUP_RUNS = 5
TIMED_RUNS  = 20
all_raw_paths = glob.glob(os.path.join(RAW_DIR, '*.jpg'))
img_path = all_raw_paths[0]

# Single-image latency
for _ in range(WARMUP_RUNS):
    predict(model, img_path)

times = []
for _ in range(TIMED_RUNS):
    t0 = time.perf_counter()
    predict(model, img_path)
    times.append(time.perf_counter() - t0)

print(f'Device: {DEVICE}')
print(f'Single-image latency ({TIMED_RUNS} runs): '
      f'avg {sum(times)/len(times)*1000:.1f} ms  '
      f'min {min(times)*1000:.1f} ms  '
      f'max {max(times)*1000:.1f} ms')
print()

# Batch throughput
print(f"{'Batch':>8}  {'Total ms':>10}  {'ms/image':>10}  {'img/sec':>10}")
print('-' * 45)
for batch_size in [1, 4, 8, 16, 32]:
    batch_paths = (all_raw_paths * ((batch_size // len(all_raw_paths)) + 1))[:batch_size]
    for _ in range(WARMUP_RUNS):
        predict_batch(model, batch_paths)
    ts = []
    for _ in range(TIMED_RUNS):
        t0 = time.perf_counter()
        predict_batch(model, batch_paths)
        ts.append(time.perf_counter() - t0)
    avg_ms = sum(ts) / len(ts) * 1000
    per_img = avg_ms / batch_size
    print(f'{batch_size:>8}  {avg_ms:>10.1f}  {per_img:>10.1f}  {1000/per_img:>10.0f}')